# Agentes con LangChain y Deep Agents

En el notebook 01 escribimos el loop del agente a mano. Acá lo hace el framework:

- **LangChain — `create_agent`**: el mismo loop, en una línea.
- **Deep Agents — `create_deep_agent`**: un harness "estilo Claude Code": planificación, archivos y subagentes.

Usamos el mismo proyecto con bugs para comparar.

## 1. Instalación

In [ ]:
%pip install -U "langchain[openai]" deepagents python-dotenv

## 2. Modelo

Con GPT-6 las tools van por la Responses API: `use_responses_api=True`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(Path("..") / ".env")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
llm = init_chat_model(OPENAI_MODEL, model_provider="openai", use_responses_api=True)
print("Modelo:", OPENAI_MODEL)

In [ ]:
from proyecto_demo import WORKSPACE, reset_workspace, run_tests

reset_workspace()
print(run_tests()[-40:])

---
# Parte A — `create_agent`

## 3. Tools con `@tool`

Ya no escribimos JSON Schema: LangChain lo arma con el **nombre**, el **docstring** y los **type hints** de la función.

In [ ]:
from langchain.tools import tool


@tool
def list_files() -> str:
    """Lista los archivos del proyecto."""
    return "\n".join(p.name for p in WORKSPACE.iterdir() if p.suffix in (".py", ".md"))


@tool
def read_file(path: str) -> str:
    """Devuelve el contenido de un archivo."""
    return (WORKSPACE / path).read_text()


@tool
def edit_file(path: str, old_str: str, new_str: str) -> str:
    """Reemplaza old_str (texto exacto) por new_str en un archivo."""
    file = WORKSPACE / path
    content = file.read_text()
    if old_str not in content:
        return "Error: old_str no está en el archivo."
    file.write_text(content.replace(old_str, new_str, 1))
    return "Archivo editado."


@tool
def correr_tests() -> str:
    """Corre los tests del proyecto y devuelve la salida."""
    return run_tests()


tools = [list_files, read_file, edit_file, correr_tests]
print(edit_file.args)

## 4. El agente

`create_agent` es el `run_agent()` del notebook 01. El resultado trae todos los mensajes: los del usuario, los del modelo (con `tool_calls`) y los resultados de las tools.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "Sos un coding agent. Leé el código antes de editarlo, "
    "corré los tests después de cada cambio y no modifiques los tests salvo que te lo pidan. "
    "Al terminar, explicá brevemente qué cambiaste."
)


def print_trace(messages):
    for msg in messages:
        for call in getattr(msg, "tool_calls", []):
            print(f"🔧 {call['name']}({str(call['args'])[:80]})")
    print("\n🤖", messages[-1].text)


agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

reset_workspace()
result = agent.invoke({"messages": [{"role": "user", "content": "Los tests fallan. Arreglá los bugs."}]})
print_trace(result["messages"])

In [ ]:
print(run_tests()[-40:])

## 5. Memoria

Con un **checkpointer** el agente recuerda la conversación. Cada conversación se identifica con un `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent_con_memoria = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "alumno-1"}}

r = agent_con_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué funciones hay en calculadora.py?"}]}, config)
print(r["messages"][-1].text)

r = agent_con_memoria.invoke({"messages": [{"role": "user", "content": "¿Cuál de esas es la más simple?"}]}, config)
print(r["messages"][-1].text)

---
# Parte B — Deep Agents

## 6. ¿Qué agrega Deep Agents?

El mismo loop, con más harness ya armado:

- **Archivos**: `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`.
- **Planificación**: `write_todos`, una lista de tareas que el agente va actualizando.
- **Subagentes**: `task`, para delegar una subtarea a otro agente con su propio contexto.

Primero, sin tocar el disco: los archivos quedan en memoria, en `result["files"]`.

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

deep_agent = create_deep_agent(model=llm, middleware=[TodoListMiddleware()])

result = deep_agent.invoke({
    "messages": [{"role": "user", "content": "Planificá con write_todos y escribí /resumen.md con 3 ideas clave sobre agentes de IA."}]
})

for todo in result["todos"]:
    print(f"[{todo['status']}] {todo['content']}")

print()
print(result["files"]["/resumen.md"]["content"])

## 7. Un deep agent sobre el proyecto

Con `FilesystemBackend` las tools de archivos trabajan sobre `workspace/` (y no pueden salir de ahí). Solo le sumamos nuestra tool `correr_tests`.

In [ ]:
from deepagents.backends import FilesystemBackend

coder = create_deep_agent(
    model=llm,
    tools=[correr_tests],
    system_prompt=SYSTEM_PROMPT,
    backend=FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True),
)

reset_workspace()
result = coder.invoke({"messages": [{"role": "user", "content": "Los tests fallan. Arreglá los bugs."}]})
print_trace(result["messages"])

In [ ]:
print(run_tests()[-40:])

## 8. Subagentes

Un subagente es otro agente con su propio prompt. El principal lo llama con la tool `task` y solo recibe el resumen, así no se le llena el contexto.

In [ ]:
test_writer = {
    "name": "test-writer",
    "description": "Escribe tests unittest para funciones nuevas.",
    "system_prompt": "Sos experto en testing. Agregá tests claros al archivo de tests existente y devolvé un resumen corto.",
}

coder = create_deep_agent(
    model=llm,
    tools=[correr_tests],
    system_prompt=SYSTEM_PROMPT,
    backend=FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True),
    subagents=[test_writer],
)

result = coder.invoke({
    "messages": [{"role": "user", "content": "Agregá mediana(numeros) en calculadora.py. Pedile al subagente test-writer que agregue sus tests en test_calculadora.py. Después corré los tests."}]
})
print_trace(result["messages"])

In [ ]:
print(run_tests()[-40:])

## Comparación

| | Desde cero (01) | `create_agent` | Deep Agents |
|---|---|---|---|
| Loop | lo escribís vos | incluido | incluido |
| Tools | JSON Schema a mano | `@tool` | trae las de archivos |
| Extras | — | memoria, middleware | TODOs, subagentes |

En TypeScript, lo mismo se hace con Vercel AI SDK (`vercel_ai_sdk/`) y Eve (`eve_coding_agent/`).

## Ejercicios

1. **Una tool nueva.** Agregá `search_code(text: str)` con `@tool`. Compará cuántas líneas te llevó contra el notebook 01.
2. **Otro subagente.** Creá un `code-reviewer` que lea el código y liste problemas sin editar. Pedile al agente que lo use antes de terminar.
3. **¿Cuánto cuesta?** Sumá `msg.usage_metadata["input_tokens"]` de los mensajes del modelo en la sección 4 y en la 7. ¿Cuál gasta más y por qué?

## Resumen

```text
create_agent      = modelo + tools + loop
create_deep_agent = create_agent + archivos + planificación + subagentes
```

El loop es siempre el del notebook 01. Lo que cambia es cuánto harness te da el framework.